In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Preprocessing 
* Pre merge EDA
* Data merge
* Post merge EDA

In [ ]:
# Read the TSV files into the DataFrame
#Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv('/Users/noimotbakare/Dropbox/Mac/Downloads/ncvoters.tsv', sep='\t')
DPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_NDPL.tsv', sep='\t')



In [ ]:
# Data distribution
counts = [14183, 9819, 98142]
labels = ["Records", "Duplicate pairs", "Non-duplicate pairs"]

plt.bar(labels, counts)
plt.title("Dataset Composition")
plt.show()

No duplicates 

In [ ]:
print(df_ncvoters['id'].duplicated().any())



In [ ]:
df_ncvoters.head()

In [ ]:
# Print the first few rows of the DataFrame
print(df_ncvoters.head())

In [ ]:
print(DPL.head())
print(NDPL.head())

In [ ]:
NDPL.info()

# Identification

Testing to ensure intersection of ids across data sets



The Hasso Plattner Institute created a unique id in snapshot: VR_Snapshot_20181106, in this data they use "id" as the identification in the ncvoters data ( not ncid or voter registration number).
The next few code boxes show an intersection between all the id variables across the data set. 

In [ ]:
# #Quick check for intersection across ids from the different tables
print(set(DPL["id1"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id1"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id1"]).intersection(set(DPL["id1"])))
print(set(DPL["id2"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id2"]).intersection(set(df_ncvoters["id"])))

In [ ]:
print(NDPL["id1"].head())
print(NDPL["id1"].tail())
print(DPL["id2"].head())
print(DPL["id2"].tail())
print(df_ncvoters["id"].head())
print(df_ncvoters["id"].tail())


In [ ]:
# Top 20 names 

# 1. Create a combined 'full_name' column
#  Use .fillna('') to ensure missing names don't break the combination
df_ncvoters['full_name'] = df_ncvoters['first_name'].fillna('').str.strip() + " " + df_ncvoters['last_name'].fillna('').str.strip()

# 2. Use value_counts() to find the most common combinations
top_20_names = df_ncvoters['full_name'].value_counts().head(20)

# 3. Display the result
print("Top 20 Most Common First + Last Names in County:")
print(top_20_names)

In [ ]:
# Checking for missing data
df_ncvoters.isna().mean().sort_values(ascending=False)

In [ ]:
missing = df_ncvoters.isna().mean().sort_values(ascending=True)

missing[missing > 0].plot.barh(figsize=(6,8))
plt.title("Field Missingness")
plt.xlabel("Proportion Missing")
plt.show()

Uniqueness and entropy -tells us how "useful" are feature is for distinguishing one person from another.

Uniqueness/cardinality of fields:

* Street name has the strongest
* Name suffix is the weakest (however we should keep suffix)

In [ ]:
# 1. Define  columns
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]

# 2. Compute nunique and ratio simultaneously
summary = df_ncvoters[cols].agg(['nunique', lambda x: x.nunique() / len(x)]).T

# 3. Rename columns and sort
summary.columns = ['unique_count', 'ratio']
summary = summary.sort_values(by='ratio', ascending=False)

print(summary)


In [ ]:
full = df_ncvoters["first_name"] + " " + df_ncvoters["last_name"]

full.value_counts().hist(bins=50)
plt.title("Full Name Frequency Distribution")
plt.xlabel("Occurrences")
plt.show()

Entropy - measure of the "uncertainty" or "disorder" of the data distribution.
How evenly is the information distributed. High entropy is better for probabilistic matching. Even if uniqueness is low, if the data is distributed evenly, it provides more "discerning power" to our algorithm.


"For an identity detection project, the ideal column should have high entropy indicating that the data is highly unique, random, and not repetitive. A high entropy value means the column provides maximum information for distinguishing between entities, while low entropy implies redundant or predictable data. 

Key Considerations for Identity Columns:
High Entropy = Better Identity Detection: Columns like User IDs, UUIDs, email addresses, or biometric hashes should have high entropy because each value is unique and unpredictable.

Low Entropy = Poor Identifier: Columns with repetitive data (e.g."Gender" or "City" in a small dataset) have low entropy, making them unsuitable as primary, unique identifiers."

In [ ]:
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]
results = []

for col in cols:
    p = df_ncvoters[col].value_counts(normalize=True)
    entropy = -(p * np.log2(p)).sum()
    results.append((col, entropy))

# sort by entropy (descending)
results.sort(key=lambda x: x[1], reverse=True)

for col, ent in results:
    print(f"{col} entropy: {ent:.4f}")

In [ ]:
df_ncvoters["full_name"].str.len().hist(bins=20)
plt.title("Full Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["first_name"].str.len().hist(bins=20)
plt.title("First Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["last_name"].str.len().hist(bins=20)
plt.title("Last Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["street_name"].str.len().hist(bins=20)
plt.title("Street Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["house_num"].hist(bins=20)
plt.title("House Number Length Distribution")
plt.show()

In [ ]:
cols = ["street_name", "last_name", "first_name", "zip_code", "mail_zipcode", "age"]

uniq = [df_ncvoters[c].nunique() / len(df_ncvoters) for c in cols]

plt.bar(cols, uniq)
plt.title("Field Uniqueness Ratio")
plt.ylabel("Unique / Total")
plt.show()

In [ ]:
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]
results = []

In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'full_name', 'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

~5.27% of suffixes are present, 94.73% are missing. Representative of real world population. Important for seperating father/son at the same address, prevents false merges. This is strong for entity resolution. 

In [ ]:
# 1. Get the raw counts
counts = df_ncvoters_frag_ID['name_sufx_cd'].value_counts(dropna=False)

# 2. Get the percentages (normalize=True) and multiply by 100
percent = df_ncvoters_frag_ID['name_sufx_cd'].value_counts(dropna=False, normalize=True) * 100

# 3. Combine them into a table
summary = pd.concat([counts, percent], axis=1)
summary.columns = ['Count', 'Percentage (%)']

print(summary)


In [ ]:
# Visualize suffix 
plt.figure(figsize=(10, 6))
ax = counts.plot(kind='bar', color='skyblue', edgecolor='black')

# Add the Percentage labels on top of the bars
for i, p in enumerate(percent):
    ax.annotate(f'{p:.1f}%', 
                (i, counts.iloc[i]), 
                ha='center', va='bottom', 
                xytext=(0, 5), textcoords='offset points',
                fontsize=10, fontweight='bold')

plt.title('Suffix Distribution (Count + %)')
plt.ylabel('Number of Records')
plt.xticks(rotation=0) 
plt.show()


# Dealing with missing observations  

The data does not have missing information in terms of nulls/NAN, it is more so empty spaces and these spaces are placeholders/important signal.

In [ ]:
#Summary Statistics 
# Generate summary for ALL variables (numeric and categorical)
summary_all = df_ncvoters_frag_ID.describe(include='all').T

# Add a 'Missing' column to show data quality for each variable
summary_all['missing'] = len(df_ncvoters_frag_ID) - summary_all['count']

# Format the table for a report (rounding decimals)
summary_all = summary_all.round(2)

print("--- Data Overview: All Variables ---")
print(summary_all)


In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
summary_all.style.background_gradient(cmap='Reds', subset=['missing'])


# Merging data 
Adding labels and merging the three data sets (ncvoter_fragmentedID, DPL and NPL). 
* df_ncvoters_frag_ID - Preprocessed data set with relevant variables.
* df_ncvoters_DPL - A list of all provided duplicates.
* df_ncvoters_NDPL - Non-duplicate pairs

# Next step Creating pairwise features 
combining duplicates and non duplicates

In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))

In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

## EDA post merges

* Checking to see if duplicates actually look similar. 
  * Class balance 
  * Label separability
  * missingness 


In [ ]:
pairs.head()

In [ ]:
#Confirming duplicates are present # Output: True
print(pairs['id_1'].duplicated().any()

)

In [ ]:
# Class balance 
pairs["label"].value_counts(normalize=True)

Same name rates, the duplicates > non-duplicates -> good seperability 

In [ ]:
#Test
pairs["same_last_name"] = (
    pairs["last_name_1"].str.lower().fillna("") ==
    pairs["last_name_2"].str.lower().fillna("")
).astype(int)


In [ ]:
pairs.groupby("label")["same_last_name"].mean()

In [ ]:
#Test
pairs["age"] = (
    pairs["age_1"] ==
    pairs["age_2"]
).astype(int)

In [ ]:
pairs.groupby("label")["age"].mean()

In [ ]:
pairs["same_address"] = (
    pairs["house_num_1"] ==
    pairs["house_num_2"]
).astype(int)

pairs.groupby("label")["same_address"].mean()

In [ ]:
pairs["same_street_name"] = (
    pairs["street_name_1"].str.lower().fillna("") ==
    pairs["street_name_2"].str.lower().fillna("")
).astype(int)

pairs.groupby("label")["same_street_name"].mean()

In [ ]:
#Overall Separability Check
from rapidfuzz.fuzz import ratio

pairs["first_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["first_name_1"]),
        str(x["first_name_2"])
    ),
    axis=1
)

pairs["last_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["last_name_1"]),
        str(x["last_name_2"])
    ),
    axis=1
)

In [ ]:
pairs.groupby("label")[["first_name_sim","last_name_sim"]].mean()

In [ ]:
#Overall Separability Check 
from sklearn.metrics import roc_auc_score

features = [
    "same_last_name",
    "same_address",
    "first_name_sim",
    "last_name_sim",
    "age"
]

for f in features:
    clean = pairs[[f, "label"]].dropna()
    auc = roc_auc_score(clean["label"], clean[f])
    print(f"{f}: AUC = {auc:.3f}")

In [ ]:
sns.kdeplot(data=pairs, x="first_name_sim", hue="label")

In [ ]:
#Hard Negatives - the cases our model might struggle with  
pairs[
    (pairs["label"] == 0) &
    (pairs["last_name_sim"] > 85)
].head(20)

In [ ]:
exact_cols = [
    "first_name",
    "last_name",
    "midl_name",
    "house_num",
    "street_name",
    "zip_code",
    "res_city_desc",
    "sex",
    "race_desc",
    "ethnic_desc"
]

for col in exact_cols:
    pairs[f"{col}_exact"] = (
        pairs[f"{col}_1"] == pairs[f"{col}_2"]
    ).astype(int)

In [ ]:
pairs["age_diff"] = abs(pairs["age_1"] - pairs["age_2"])
pairs["age_exact"] = (pairs["age_diff"] == 0).astype(int)
pairs["age_close"] = (pairs["age_diff"] <= 1).astype(int)
pairs.head(50)

Creating binary columns for names that have the same phonetic pronounciation - Real duplicates often differ by spelling but not pronunciation. Phonetic match = strong evidence of same person.

In [ ]:
import jellyfish

def phonetic_features(pairs, col):

   pairs[f"{col}_soundex_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.soundex)
   pairs[f"{col}_soundex_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.soundex)

   pairs[f"{col}_metaphone_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.metaphone)
   pairs[f"{col}_metaphone_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.metaphone)

   pairs[f"{col}_soundex_match"] = (
        pairs[f"{col}_soundex_1"] == pairs[f"{col}_soundex_2"]
    ).astype(int)

   pairs[f"{col}_metaphone_match"] = (
        pairs[f"{col}_metaphone_1"] == pairs[f"{col}_metaphone_2"]
    ).astype(int)

   pairs.head(50)

In [ ]:
for col in ["first_name", "last_name"]:
    phonetic_features(pairs, col)

In [ ]:
#Missing agreement future 
for col in ["midl_name", "phone_num"]:
    pairs[f"{col}_both_missing"] = (
        (pairs[f"{col}_1"] == "") &
        (pairs[f"{col}_2"] == "")
    ).astype(int)

In [ ]:
pairs.groupby("label")[[
    "first_name_soundex_match",
    "last_name_soundex_match"
]].mean()

In [ ]:
pairs.groupby("label")[[
    "first_name_metaphone_match",
    "last_name_metaphone_match"
]].mean()

These are the features we will use to build Siamese network and the features that will be used for modeling, respectively.

In [ ]:
# ========== SIAMESE NETWORK FEATURES ==========
# Raw text fields that will be embedded
siamese_features = [
    # Names
    'first_name_1', 'first_name_2',
    'midl_name_1', 'midl_name_2', 
    'last_name_1', 'last_name_2',
    'name_sufx_cd_1', 'name_sufx_cd_2',
    
    # Address
    'house_num_1', 'house_num_2',
    'street_name_1', 'street_name_2',
    'street_type_cd_1', 'street_type_cd_2',
    'unit_num_1', 'unit_num_2',
    'zip_code_1', 'zip_code_2',
    'res_city_desc_1', 'res_city_desc_2',
    
    # Demographics
    'age_1', 'age_2',
    'phone_num_1', 'phone_num_2',
    'sex_1', 'sex_2',
    'race_desc_1', 'race_desc_2'
]

# ========== BASELINE MODEL FEATURES ==========
# Engineered comparison features for traditional ML
model_features = [
    # Exact matches
    'same_last_name', 'same_street_name',
    'first_name_exact', 'last_name_exact', 'midl_name_exact',
    'house_num_exact', 'street_name_exact', 'zip_code_exact',
    'res_city_desc_exact', 'sex_exact', 'race_desc_exact',
    # Similarity scores
    'first_name_sim', 'last_name_sim', 
    
    # Age comparisons
    'age_diff', 'age_exact', 'age_close',
    
    # Phonetic matches
    'first_name_soundex_match', 'first_name_metaphone_match',
    'last_name_soundex_match', 'last_name_metaphone_match',
    
    # Missingness indicators
    'midl_name_both_missing', 'phone_num_both_missing'
]

# ========== METADATA (for tracking, not training) ==========
metadata_cols = ['id1', 'id2', 'label', 'participation']

print(f"Siamese features: {len(siamese_features)}")
print(f"Baseline model features: {len(model_features)}")

In [ ]:
print(pairs.columns)